# 🔍 Cross-Validation K-Fold e Interpretabilidad
## DecisionTree, RandomForest y XGBoost

**Objetivo educativo:** entender cómo la validación cruzada no solo mejora la **estimación del desempeño**, sino que también **mejora la interpretabilidad** del modelo al revelar qué patrones son **estables** entre distintas particiones de los datos.

### 💡 La idea central

Un modelo entrenado con un único split train/test puede darnos importancias de variables que **parecen** convincentes pero son **ruido de esa partición particular**. Al entrenar el mismo modelo en K particiones distintas y agregar las importancias, distinguimos:

- **Señal** → variables importantes en **todos** los folds → interpretación confiable.
- **Ruido** → variables importantes solo en **algunos** folds → alerta amarilla.

### Contenido:
1. Setup y dataset (Wine — clasificación multiclase).
2. **La trampa del split único**: cómo cambian las conclusiones con diferentes semillas.
3. **K-Fold estratificado** con los tres modelos.
4. **Análisis 1**: distribución de métricas (boxplot por fold).
5. **Análisis 2**: estabilidad de feature importances (media ± desviación).
6. **Análisis 3**: ranking stability.
7. **Análisis 4**: consistencia estructural del árbol de decisión.
8. **Análisis 5**: consenso entre modelos.
9. Comparativa final y conclusiones.


## 1. Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import warnings
warnings.filterwarnings('ignore')

from sklearn.datasets import load_wine
from sklearn.model_selection import (train_test_split, StratifiedKFold,
                                       cross_val_score, RepeatedStratifiedKFold)
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import (accuracy_score, precision_score, recall_score, f1_score)
from sklearn.base import clone
from xgboost import XGBClassifier
from collections import Counter

SEED = 42
np.random.seed(SEED)
sns.set_style('whitegrid')
plt.rcParams['figure.dpi'] = 90
print("✅ Setup listo")

## 2. Dataset: Wine (clasificación multi-clase)

Elegimos el dataset **Wine** por tres razones didácticas:
- **Muestras limitadas (178)** → un único split train/test es muy sensible a la semilla, lo que amplifica el problema que queremos ilustrar.
- **13 features con correlaciones internas** (grupos de fenoles, propiedades químicas) → algunas features serán intercambiables entre folds.
- **3 clases balanceadas** → nos concentramos en el objetivo CV+interpretabilidad sin distracciones de desbalance.


In [ ]:
data = load_wine(as_frame=True)
df = data.frame
X = df.drop(columns='target').values
y = df['target'].values
feature_names = list(data.feature_names)

print(f"Muestras: {len(y)}  |  Features: {len(feature_names)}  |  Clases: {len(np.unique(y))}")
print(f"\nBalance de clases:")
print(pd.Series(y).value_counts().sort_index().rename('n'))
df.head()

## 3. La trampa del split único

Antes de aplicar CV, vamos a exponer un problema. Entrenaremos los mismos tres modelos con **cinco semillas distintas** para `train_test_split`. En un mundo ideal, la accuracy y las importancias deberían ser casi iguales. Veremos que no lo son.


In [ ]:
semillas = [0, 1, 42, 100, 2024]
modelos_prueba = {
    'DT':  DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'RF':  RandomForestClassifier(n_estimators=100, max_depth=6, random_state=SEED, n_jobs=-1),
    'XGB': XGBClassifier(n_estimators=100, max_depth=4, learning_rate=0.1,
                          eval_metric='mlogloss', random_state=SEED, n_jobs=-1),
}

resultados_semilla    = []
importancias_semilla  = {n: [] for n in modelos_prueba}

for seed in semillas:
    X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25,
                                               stratify=y, random_state=seed)
    for nombre, modelo in modelos_prueba.items():
        m = clone(modelo)
        m.fit(X_tr, y_tr)
        acc = accuracy_score(y_te, m.predict(X_te))
        resultados_semilla.append({'Modelo': nombre, 'Semilla': seed, 'Accuracy': acc})
        importancias_semilla[nombre].append(m.feature_importances_)

df_seed = pd.DataFrame(resultados_semilla).pivot(index='Semilla', columns='Modelo', values='Accuracy')
print("Accuracy según semilla del split:")
df_seed.round(3)

In [ ]:
# Rango de variabilidad
rango = df_seed.max() - df_seed.min()
std   = df_seed.std()
print("Rango (max − min) de accuracy entre semillas:")
print(rango.round(3))
print("\nDesviación estándar:")
print(std.round(3))

In [ ]:
# ¿Y las importancias? El caso más ilustrativo es el DecisionTree
df_imp_dt = pd.DataFrame(importancias_semilla['DT'], columns=feature_names, index=semillas)
df_imp_rf = pd.DataFrame(importancias_semilla['RF'], columns=feature_names, index=semillas)

fig, axes = plt.subplots(2, 1, figsize=(14, 8))

df_imp_dt.T.plot(kind='bar', ax=axes[0], width=0.85, colormap='Set2', edgecolor='black')
axes[0].set_title('DecisionTree — importancias por semilla de split (dramáticamente inestables)')
axes[0].set_ylabel('Importancia')
axes[0].legend(title='Semilla', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[0].tick_params(axis='x', rotation=45)

df_imp_rf.T.plot(kind='bar', ax=axes[1], width=0.85, colormap='Set2', edgecolor='black')
axes[1].set_title('Random Forest — importancias por semilla de split (más estables, pero los valores aún varían)')
axes[1].set_ylabel('Importancia')
axes[1].legend(title='Semilla', bbox_to_anchor=(1.02, 1), loc='upper left')
axes[1].tick_params(axis='x', rotation=45)

plt.tight_layout(); plt.show()

print("Feature más importante según cada semilla:")
print(pd.DataFrame({
    'DT top':  df_imp_dt.idxmax(axis=1),
    'RF top':  df_imp_rf.idxmax(axis=1)
}))

### 🚨 Observación clave

Con solo **cambiar la semilla del split**, obtenemos:
- Accuracies distintas para el mismo modelo con los mismos hiperparámetros (dramático en DT, sutil en RF).
- **La feature más importante puede cambiar** entre semillas — especialmente para el DecisionTree.
- Incluso cuando el top-1 es estable (Random Forest), **los valores numéricos varían** y una feature "importante" con una semilla puede caer al fondo con otra.

Si reportáramos las conclusiones con un solo split, estaríamos **contando historias diferentes** dependiendo del capricho de una semilla. Esto motiva el uso de **cross-validation**: agregamos K historias en una **narrativa robusta con incertidumbre cuantificada**.


## 4. K-Fold Cross-Validation estratificado

**StratifiedKFold** divide el dataset en K partes preservando la proporción de clases en cada fold. Entrenamos K veces, usando cada parte una vez como test.

```
Fold 1: [TEST ][           TRAIN            ]
Fold 2: [    ][TEST][         TRAIN          ]
Fold 3: [       TRAIN     ][TEST][   TRAIN   ]
Fold 4: [         TRAIN            ][TEST][ T ]
Fold 5: [           TRAIN            ][TEST  ]
```

Cada muestra se usa **exactamente una vez** como test, y siempre **K−1 veces** como parte del train.


In [ ]:
K = 5
kf = StratifiedKFold(n_splits=K, shuffle=True, random_state=SEED)

modelos_cv = {
    'Decision Tree': DecisionTreeClassifier(max_depth=4, random_state=SEED),
    'Random Forest': RandomForestClassifier(n_estimators=200, max_depth=6,
                                             random_state=SEED, n_jobs=-1),
    'XGBoost':       XGBClassifier(n_estimators=200, max_depth=4, learning_rate=0.1,
                                    eval_metric='mlogloss', random_state=SEED, n_jobs=-1),
}

# Estructuras para acumular resultados
metricas_cv     = {n: {'acc': [], 'precision': [], 'recall': [], 'f1': []} for n in modelos_cv}
importancias_cv = {n: [] for n in modelos_cv}
raices_dt       = []   # feature en la raíz del DecisionTree por fold

for fold, (tr_idx, te_idx) in enumerate(kf.split(X, y)):
    X_tr_f, X_te_f = X[tr_idx], X[te_idx]
    y_tr_f, y_te_f = y[tr_idx], y[te_idx]
    
    for nombre, modelo in modelos_cv.items():
        m = clone(modelo)
        m.fit(X_tr_f, y_tr_f)
        y_pred = m.predict(X_te_f)
        
        metricas_cv[nombre]['acc'].append(accuracy_score(y_te_f, y_pred))
        metricas_cv[nombre]['precision'].append(precision_score(y_te_f, y_pred, average='macro'))
        metricas_cv[nombre]['recall'].append(recall_score(y_te_f, y_pred, average='macro'))
        metricas_cv[nombre]['f1'].append(f1_score(y_te_f, y_pred, average='macro'))
        importancias_cv[nombre].append(m.feature_importances_)
        
        if nombre == 'Decision Tree':
            raices_dt.append(feature_names[m.tree_.feature[0]])
    
    print(f"  ✓ Fold {fold+1}/{K} completado")

print("\n✅ K-Fold CV finalizado")

### 💡 Atajo: `cross_val_score`

Sklearn ofrece un atajo cuando solo quieres las métricas (sin importancias por fold):


In [ ]:
# Atajo equivalente para métricas simples
for nombre, modelo in modelos_cv.items():
    scores = cross_val_score(clone(modelo), X, y, cv=kf, scoring='accuracy', n_jobs=-1)
    print(f"{nombre:15s} → Accuracy: {scores.mean():.3f} ± {scores.std():.3f}")

## 5. Análisis 1 · Distribución de métricas

En lugar de un único número (que puede estar sesgado por una partición afortunada), tenemos una **distribución** de valores. La media y la desviación estándar son mucho más informativas que un punto único.


In [ ]:
resumen = []
for nombre, metricas in metricas_cv.items():
    resumen.append({
        'Modelo': nombre,
        'Accuracy (μ ± σ)':  f"{np.mean(metricas['acc']):.3f} ± {np.std(metricas['acc']):.3f}",
        'F1 macro (μ ± σ)':  f"{np.mean(metricas['f1']):.3f} ± {np.std(metricas['f1']):.3f}",
        'Precision (μ)':     round(np.mean(metricas['precision']), 3),
        'Recall (μ)':        round(np.mean(metricas['recall']), 3),
    })
pd.DataFrame(resumen).set_index('Modelo')

In [ ]:
# Boxplot de accuracy y F1 por modelo
fig, axes = plt.subplots(1, 2, figsize=(13, 4.5))

for metric_key, ax, titulo in [('acc', axes[0], 'Accuracy'), ('f1', axes[1], 'F1 macro')]:
    data_bp = [metricas_cv[n][metric_key] for n in modelos_cv]
    bp = ax.boxplot(data_bp, labels=list(modelos_cv.keys()), patch_artist=True, widths=0.5)
    for patch, color in zip(bp['boxes'], ['#ff9999', '#66b3ff', '#99ff99']):
        patch.set_facecolor(color)
    # Puntos individuales por fold
    for i, values in enumerate(data_bp):
        ax.scatter(np.full(len(values), i+1) + np.random.uniform(-0.08, 0.08, len(values)),
                   values, color='black', alpha=0.7, s=40, zorder=3)
    ax.set_title(f'{titulo} por fold ({K}-fold CV)')
    ax.set_ylabel(titulo); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

### 📖 Cómo leer los boxplots
- **Caja pequeña** → modelo estable entre folds (baja varianza de desempeño).
- **Caja grande** → modelo sensible a la partición (poca confianza en un valor puntual).
- **Puntos negros** → valores individuales por fold. Puntos fuera de la caja indican folds "difíciles" que podrían valer la pena investigar.

Prefiere un modelo con **caja pequeña**, aunque su mediana sea ligeramente inferior — la consistencia importa más en producción.


## 6. Análisis 2 · Estabilidad de feature importances

Ahora viene lo interesante: **¿qué variables son importantes de manera consistente entre folds?** Esta es la clave de una interpretabilidad robusta.


In [ ]:
# Convertimos importancias a DataFrames: filas = folds, columnas = features
df_imp = {n: pd.DataFrame(v, columns=feature_names) for n, v in importancias_cv.items()}

# Resumen tabular: μ ± σ por feature y modelo
resumen_imp = pd.DataFrame({
    n: df_imp[n].mean().round(3).astype(str) + ' ± ' + df_imp[n].std().round(3).astype(str)
    for n in modelos_cv
})
resumen_imp

In [ ]:
# Bar chart con error bars (μ ± σ) para cada modelo
fig, axes = plt.subplots(3, 1, figsize=(14, 11))

for ax, (nombre, dfi) in zip(axes, df_imp.items()):
    means = dfi.mean()
    stds  = dfi.std()
    orden = means.sort_values(ascending=False).index
    ax.bar(range(len(orden)), means[orden], yerr=stds[orden], capsize=4,
           color='steelblue', alpha=0.85, edgecolor='black', error_kw={'ecolor': 'crimson', 'linewidth': 1.5})
    ax.set_xticks(range(len(orden)))
    ax.set_xticklabels(orden, rotation=45, ha='right')
    ax.set_title(f'{nombre} — importancia (μ ± σ entre {K} folds)')
    ax.set_ylabel('Importancia'); ax.grid(axis='y', alpha=0.3)

plt.tight_layout(); plt.show()

### 💡 Cómo interpretar

- **Barra alta + error corto** → feature realmente importante (señal clara).
- **Barra alta + error largo** → feature "a veces importante" — cuidado al concluir.
- **Barra baja + error corto** → feature confiablemente poco útil (candidata a descartar).
- **Barra baja + error largo** → feature que a veces salta pero suele no ser útil.

La regla es: **mira la barra Y el error juntos**. Un solo split no te muestra el error.


In [ ]:
# Heatmap comparativo: importancia por fold para los 3 modelos
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nombre, dfi) in zip(axes, df_imp.items()):
    orden = dfi.mean().sort_values(ascending=False).index
    dfi_ord = dfi[orden].T   # filas = features, columnas = folds
    sns.heatmap(dfi_ord, cmap='YlOrRd', ax=ax, cbar_kws={'label': 'Importancia'},
                linewidths=0.4, linecolor='white')
    ax.set_title(f'{nombre}')
    ax.set_xlabel('Fold'); ax.set_ylabel('')
    ax.set_xticklabels([f'F{i+1}' for i in range(K)], rotation=0)

plt.suptitle(f'Heatmap de importancias por fold\n(filas = features ordenadas por μ; columnas = folds)', y=1.03)
plt.tight_layout(); plt.show()

## 7. Análisis 3 · Estabilidad del ranking

Otro ángulo: en lugar de comparar **valores** de importancia (que pueden estar en escalas distintas entre modelos), comparamos **rankings**. Una feature realmente crítica estará entre las **top-K** en todos los folds.


In [ ]:
# Rankings por fold: 1 = más importante
rankings = {n: dfi.rank(axis=1, ascending=False, method='min') for n, dfi in df_imp.items()}

# Distribución del ranking para las top-8 features de cada modelo
fig, axes = plt.subplots(1, 3, figsize=(18, 5))
for ax, (nombre, ranks) in zip(axes, rankings.items()):
    orden = ranks.mean().sort_values().index[:8]   # top-8 por ranking medio
    data_plot = [ranks[f].values for f in orden]
    bp = ax.boxplot(data_plot, labels=orden, patch_artist=True, vert=False, widths=0.6)
    for patch in bp['boxes']:
        patch.set_facecolor('lightblue')
    ax.set_title(f'{nombre} — ranking de top-8 features')
    ax.set_xlabel('Ranking (1 = más importante)')
    ax.invert_xaxis()          # rank 1 a la derecha
    ax.grid(axis='x', alpha=0.3)

plt.tight_layout(); plt.show()

### 📖 Cómo leer estos boxplots de ranking

- **Caja estrecha y a la derecha (ranks 1-2)** → feature consistentemente crítica → **interpretación segura**.
- **Caja ancha** → su importancia relativa varía entre folds → sospechosa: puede intercambiarse con otras features correlacionadas.
- **Caja estrecha pero a la izquierda** → consistentemente poco importante → candidata clara a descartar.

Este análisis captura algo que la media/std no captura: **la ordenación relativa** entre features. Dos features pueden tener importancias medias similares pero rankings muy distintos si son correlacionadas y se "roban" el crédito entre folds.


## 8. Análisis 4 · Consistencia estructural del árbol de decisión

En un **DecisionTree**, la feature en la **raíz** es la que mayor información aporta según el criterio de división. Con CV podemos ver **qué feature aparece en la raíz en cada fold** — un indicador estructural, no solo estadístico, de importancia.


In [ ]:
print("Feature en la raíz del DecisionTree por fold:")
for i, r in enumerate(raices_dt):
    print(f"  Fold {i+1}: {r}")

frecuencia_raiz = pd.Series(raices_dt).value_counts()
print(f"\nFrecuencia de aparición como raíz (de {K} folds):")
print(frecuencia_raiz)

if frecuencia_raiz.iloc[0] == K:
    print(f"\n🎯 La feature '{frecuencia_raiz.index[0]}' es la raíz en TODOS los folds → señal estructural muy fuerte.")
else:
    print(f"\n⚠️  Hay competencia por la raíz — múltiples features son casi equivalentes en información.")

In [ ]:
# Referencia visual: árbol entrenado con TODOS los datos (para que veas la estructura)
dt_ref = DecisionTreeClassifier(max_depth=3, random_state=SEED)
dt_ref.fit(X, y)

plt.figure(figsize=(18, 7))
plot_tree(dt_ref, feature_names=feature_names, class_names=list(data.target_names),
          filled=True, rounded=True, fontsize=10, impurity=False)
plt.title('Árbol de decisión de referencia (entrenado en todo el dataset, max_depth=3)')
plt.tight_layout(); plt.show()

## 9. Análisis 5 · Consenso entre modelos

Si los **tres modelos** coinciden en qué features son importantes → tenemos una **narrativa muy sólida** sobre el dataset. El consenso entre modelos con arquitecturas distintas es el mejor validador de que estamos captando señal real.


In [ ]:
# Top-5 por modelo (según importancia media en CV)
top_por_modelo = {n: dfi.mean().sort_values(ascending=False).head(5).index.tolist()
                  for n, dfi in df_imp.items()}

for n, t in top_por_modelo.items():
    print(f"{n:15s}: {t}")

# Consenso: features que aparecen en el top-5 de ≥ 2 modelos
todas = [f for lista in top_por_modelo.values() for f in lista]
conteo = Counter(todas)
consenso_parcial = sorted([f for f, c in conteo.items() if c >= 2], key=lambda x: -conteo[x])
consenso_total   = [f for f, c in conteo.items() if c == 3]

print(f"\n✅ En el top-5 de ≥ 2 modelos: {consenso_parcial}")
print(f"🎯 En el top-5 de los 3 modelos: {consenso_total}")

In [ ]:
# Gráfico comparativo: top features y sus importancias por modelo
comparativa = pd.DataFrame({n: dfi.mean() for n, dfi in df_imp.items()})
top_globales = comparativa.mean(axis=1).sort_values(ascending=False).head(8).index
comp_top = comparativa.loc[top_globales]

fig, ax = plt.subplots(figsize=(11, 5.5))
comp_top.plot(kind='bar', ax=ax, width=0.8, colormap='Set2', edgecolor='black')
ax.set_title(f'Top-8 features por importancia media en {K}-fold CV\n(comparadas entre los 3 modelos)')
ax.set_ylabel('Importancia media entre folds')
plt.xticks(rotation=45, ha='right')
plt.legend(title='Modelo', loc='upper right')
plt.tight_layout(); plt.show()

## 10. Comparativa final: single-split vs K-Fold CV


In [ ]:
# Baseline con la semilla 42
X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)

comparacion = []
for nombre, modelo in modelos_cv.items():
    m_baseline = clone(modelo)
    m_baseline.fit(X_tr, y_tr)
    acc_single = accuracy_score(y_te, m_baseline.predict(X_te))
    
    acc_cv_mean = np.mean(metricas_cv[nombre]['acc'])
    acc_cv_std  = np.std(metricas_cv[nombre]['acc'])
    
    dentro = 'Sí' if abs(acc_single - acc_cv_mean) < 2*acc_cv_std else 'No'
    comparacion.append({
        'Modelo': nombre,
        'Single-split (seed=42)':  round(acc_single, 3),
        'CV μ ± σ':                f"{acc_cv_mean:.3f} ± {acc_cv_std:.3f}",
        'CV min':                  round(min(metricas_cv[nombre]['acc']), 3),
        'CV max':                  round(max(metricas_cv[nombre]['acc']), 3),
        '¿Single dentro de μ ± 2σ?': dentro
    })

pd.DataFrame(comparacion).set_index('Modelo')

In [ ]:
# Visualización: valor del single-split superpuesto a la distribución CV
fig, ax = plt.subplots(figsize=(11, 5))

X_tr, X_te, y_tr, y_te = train_test_split(X, y, test_size=0.25, stratify=y, random_state=SEED)
data_plot, single_vals = [], []
for nombre, modelo in modelos_cv.items():
    m_baseline = clone(modelo); m_baseline.fit(X_tr, y_tr)
    single_vals.append(accuracy_score(y_te, m_baseline.predict(X_te)))
    data_plot.append(metricas_cv[nombre]['acc'])

bp = ax.boxplot(data_plot, labels=list(modelos_cv.keys()), patch_artist=True, widths=0.5)
for patch, color in zip(bp['boxes'], ['#ff9999', '#66b3ff', '#99ff99']):
    patch.set_facecolor(color)

# Superponemos el valor del single-split
for i, v in enumerate(single_vals):
    ax.scatter(i+1, v, color='crimson', s=200, marker='*', zorder=5,
               edgecolors='black', linewidth=1.5, label='Single-split (seed=42)' if i==0 else None)

ax.set_title(f'Distribución CV ({K} folds) vs valor puntual del single-split')
ax.set_ylabel('Accuracy'); ax.legend(); ax.grid(axis='y', alpha=0.3)
plt.tight_layout(); plt.show()

## 📋 Conclusiones · ¿Cómo mejoró CV la interpretabilidad?

| # | Beneficio | Contribución a la interpretabilidad |
|---|---|---|
| 1 | Certidumbre de métricas | Reportar `μ ± σ` es más honesto que un punto único — sabemos qué tan replicable es el resultado. |
| 2 | Certidumbre de importancias | Barras con error nos dicen cuánto variarían nuestras conclusiones ante otro dataset del mismo problema. |
| 3 | Ranking estable | El boxplot de rankings distingue features "casi siempre top" de las "a veces sí, a veces no". |
| 4 | Consenso estructural (DT) | Ver la misma feature en la raíz de todos los folds es evidencia estructural, no solo estadística. |
| 5 | Consenso entre modelos | Cuando DT, RF y XGBoost coinciden → señal muy fuerte que no se ve con un solo split. |
| 6 | Folds difíciles | Outliers en los boxplots indican particiones problemáticas que valen la pena investigar. |

### 🎯 Regla de oro

> Si un patrón (importancia, ranking, división) aparece en **todos los folds**, es **señal**.
> Si aparece solo en **algunos folds**, es **ruido** disfrazado de señal.

### 💭 Preguntas para discusión

1. En el análisis, **¿qué feature quedó en el top-5 de los tres modelos?** ¿Coincide con la feature en la raíz del DT?
2. Si dos features son fuertemente correlacionadas, ¿qué esperarías ver en el boxplot de rankings de ambas?
3. ¿Por qué reportar `μ ± σ` es más útil que reportar solo la media al comparar modelos?
4. ¿Cuándo NO conviene usar K-Fold CV? (Pista: series temporales, dataset enorme, cómputo restringido).

### 🧪 Ejercicios propuestos

1. **Aumenta K a 10.** ¿Se estrechan las cajas de las importancias? ¿Cambian los rankings top?
2. **Usa `RepeatedStratifiedKFold(n_splits=5, n_repeats=10)`** — 50 estimaciones en lugar de 5. Compara la anchura de las cajas.
3. **Usa `permutation_importance`** de sklearn en cada fold en lugar de `feature_importances_`. ¿Los rankings coinciden con los de la importancia interna?
4. **Aplica el análisis al dataset breast cancer** de los notebooks anteriores. ¿Cuáles features son las estructuralmente importantes ahí?
5. **Agrega ruido** artificial (features aleatorias) al dataset. ¿Las features ruidosas quedan al fondo del ranking en todos los folds, como esperaríamos?
